# EEEM071 — Vehicle Re-ID Training

Runs in **Google Colab** (mounts Drive, clones repo, installs deps) and **locally** (uses repo working directory). Environment is detected automatically via `in_colab()`.

Parameterised with [Papermill](https://papermill.readthedocs.io). Pre-configured experiment notebooks live in `notebooks/`. Run the full LR sweep with:
```bash
python scripts/run_lr_sweep.py
```

In [ ]:
# ── Student info ──────────────────────────────────────────────────────────────
student_id   = "your_student_id"
student_name = "Your_Name"

# ── Dataset ───────────────────────────────────────────────────────────────────
source_dataset = "veri"         # dataset used for training
target_dataset = "veri"         # dataset used for evaluation
data_root      = "../data/VeRi" # local path; overridden automatically in Colab

# ── Colab settings (ignored when running locally) ─────────────────────────────
colab_repo_url        = "https://github.com/Surrey-EEEM071-CVDL/Surrey_EEEM071_Coursework.git"
colab_repo_dir        = "/content/Surrey_EEEM071_Coursework"
colab_drive_data_path = "/content/drive/MyDrive/VeRi"

# ── Model ─────────────────────────────────────────────────────────────────────
arch          = "mobilenet_v3_small"  # resnet50 | resnet18 | mobilenet_v3_small | vgg16
no_pretrained = False                 # set True to skip ImageNet pretrained weights

# ── Image dimensions ──────────────────────────────────────────────────────────
image_height = 224
image_width  = 224

# ── Optimisation ──────────────────────────────────────────────────────────────
optimizer     = "amsgrad"  # adam | amsgrad | sgd | rmsprop
learning_rate = 0.0001
weight_decay  = 5e-4
max_epochs    = 1
stepsize      = "20 40"   # space-separated LR decay milestones
gamma         = 0.1       # LR decay factor
seed          = 1

# ── Batch sizes ───────────────────────────────────────────────────────────────
train_batch_size = 64
test_batch_size  = 100

# ── Loss ──────────────────────────────────────────────────────────────────────
margin      = 0.3  # triplet loss margin
lambda_xent = 1.0  # cross-entropy loss weight
lambda_htri = 1.0  # hard-triplet loss weight
label_smooth = False

# ── Augmentation ──────────────────────────────────────────────────────────────
random_erase = False
color_jitter = False
color_aug    = False

# ── Hardware & output ─────────────────────────────────────────────────────────
gpu_devices   = "0"                      # e.g. "0" or "0,1"
use_cpu       = False
eval_freq     = -1                       # -1 = evaluate only at end
evaluate_only = False                    # skip training, run eval only
save_dir      = "logs/mobilenet_v3_small-veri-lr1e-4"

## 1. Environment Detection

In [ ]:
import json
import os
import re
import subprocess
import sys
import time
from collections import defaultdict
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt


def in_colab() -> bool:
    """Return True when running inside Google Colab."""
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False


IS_COLAB = in_colab()
print(f"Environment : {'Google Colab' if IS_COLAB else 'local'}")
print(f"Python      : {sys.executable}")

## 2a. Colab Setup

> **Colab checklist before running:**
> 1. *Edit → Notebook settings* → set Hardware accelerator to **GPU**
> 2. Click the folder icon on the left and **mount Google Drive** (or wait for the cell below to prompt you)
> 3. Upload `VeRi.zip` to `My Drive/` and note the path — set `colab_drive_data_path` in the parameters cell
>
> *Skipped automatically when not in Colab.*

In [ ]:
if IS_COLAB:
    # ── Step 1: GPU check ────────────────────────────────────────────────────
    result = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
    if result.returncode == 0:
        print(result.stdout)
    else:
        print("WARNING: nvidia-smi not available. Make sure GPU is enabled in Colab settings.")

    # ── Step 2: Mount Google Drive ───────────────────────────────────────────
    from google.colab import drive
    drive.mount("/content/drive")

    # ── Step 3: Clone / update repo ──────────────────────────────────────────
    if not Path(colab_repo_dir).exists():
        subprocess.run(
            ["git", "clone", colab_repo_url, colab_repo_dir],
            check=True,
        )
        print(f"Cloned repo -> {colab_repo_dir}")
    else:
        print(f"Repo already present at {colab_repo_dir}")

    os.chdir(colab_repo_dir)
    print(f"Working directory: {Path.cwd()}")

    # ── Step 4: Install gdown ────────────────────────────────────────────────
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-U", "--no-cache-dir", "gdown", "--pre"],
        check=True,
    )

    # ── Step 5: Data note ────────────────────────────────────────────────────
    # Download datasets from the assignment doc link, upload to Google Drive,
    # then set colab_drive_data_path to point at the extracted folder.
    if Path("/content/VeRi.zip").exists():
        print("VeRi.zip found in /content — unzip manually if not already done.")
    print(f"Data root (Colab): {colab_drive_data_path}")

    # Override data_root for all subsequent cells
    data_root = colab_drive_data_path

else:
    print("Colab setup skipped (local environment)")

## 2b. Local Setup

*Skipped automatically when running in Colab.*

In [ ]:
if not IS_COLAB:
    # Normalise working directory to the folder containing this notebook
    # so that main.py and src/ are importable regardless of where papermill
    # was invoked from.
    notebook_dir = Path(globals().get("__vsc_ipynb_file__", "")).parent
    if notebook_dir.exists() and notebook_dir != Path("."):
        os.chdir(notebook_dir)
    # Fallback: if this file is in notebooks/, step up one level to repo root
    if not Path("main.py").exists() and Path("../main.py").exists():
        os.chdir("..")
    print(f"Working directory: {Path.cwd()}")

    # GPU check (non-fatal)
    try:
        result = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
        if result.returncode == 0:
            print("\n".join(result.stdout.splitlines()[:10]))
        else:
            print("nvidia-smi not available — training will use CPU")
    except FileNotFoundError:
        print("nvidia-smi not found — training will use CPU")

else:
    print("Local setup skipped (Colab environment)")

print(f"\nStudent  : {student_name} ({student_id})")
print(f"Arch     : {arch}")
print(f"Dataset  : {source_dataset} -> {target_dataset}")
print(f"Data     : {data_root}")
print(f"Epochs   : {max_epochs}  |  LR: {learning_rate}  |  Optim: {optimizer}")

## 3. Build Training Command

In [ ]:
cmd = [
    sys.executable, "main.py",
    "-s", source_dataset,
    "-t", target_dataset,
    "-a", arch,
    "--root",             data_root,
    "--height",           str(image_height),
    "--width",            str(image_width),
    "--optim",            optimizer,
    "--lr",               str(learning_rate),
    "--weight-decay",     str(weight_decay),
    "--max-epoch",        str(max_epochs),
    "--stepsize",         *stepsize.split(),
    "--gamma",            str(gamma),
    "--train-batch-size", str(train_batch_size),
    "--test-batch-size",  str(test_batch_size),
    "--margin",           str(margin),
    "--lambda-xent",      str(lambda_xent),
    "--lambda-htri",      str(lambda_htri),
    "--seed",             str(seed),
    "--eval-freq",        str(eval_freq),
    "--save-dir",         save_dir,
    "--gpu-devices",      gpu_devices,
]

# Boolean flags
if label_smooth:  cmd.append("--label-smooth")
if random_erase:  cmd.append("--random-erase")
if color_jitter:  cmd.append("--color-jitter")
if color_aug:     cmd.append("--color-aug")
if no_pretrained: cmd.append("--no-pretrained")
if use_cpu:       cmd.append("--use-cpu")
if evaluate_only: cmd.append("--evaluate")

# Inject student credentials via environment variables
env = os.environ.copy()
env["STUDENT_ID"]   = student_id
env["STUDENT_NAME"] = student_name

# Print shell equivalent for reproducibility
shell_cmd = (
    f"STUDENT_ID='{student_id}' STUDENT_NAME='{student_name}' "
    + " ".join(cmd)
)
print(f"Shell equivalent:\n{shell_cmd}")

## 4. Run Training

In [ ]:
t0 = time.time()
_output_lines = []  # captured for metric parsing in the next cell

process = subprocess.Popen(
    cmd,
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

for line in process.stdout:
    print(line, end="", flush=True)
    _output_lines.append(line)

process.wait()
elapsed = time.time() - t0

if process.returncode != 0:
    raise RuntimeError(f"main.py exited with return code {process.returncode}")

print(f"\nTraining complete -- {elapsed/60:.1f} min ({elapsed:.0f} s)")

## 5. Metrics Visualisation

Parse training logs and evaluation results, then plot training curves and CMC metrics.

In [ ]:
# ── Log format reference ─────────────────────────────────────────────────────
# Training batch:  "Epoch: [E][B/N]\t...\tXent x.x (avg)\tHtri x.x (avg)\tAcc x.x (avg)"
# Evaluation mAP:  "mAP: XX.X%"
# CMC entry:       "Rank-1  : XX.X%"

BATCH_RE = re.compile(
    r"Epoch: \[(\d+)\]\[(\d+)/(\d+)\]"
    r".*Xent [\d.]+ \(([\d.]+)\)"
    r".*Htri [\d.]+ \(([\d.]+)\)"
    r".*Acc [\d.]+ \(([\d.]+)\)"
)
MAP_RE  = re.compile(r"mAP: ([\d.]+)%")
RANK_RE = re.compile(r"Rank-(\d+)\s*:\s*([\d.]+)%")

batch_stats = []  # [{epoch, batch, n_batches, xent, htri, acc}, ...]
final_map   = None
cmc         = {}  # {rank_int: pct_float}

for line in _output_lines:
    m = BATCH_RE.search(line)
    if m:
        epoch, batch, n_batches, xent, htri, acc = m.groups()
        batch_stats.append({
            "epoch": int(epoch), "batch": int(batch), "n_batches": int(n_batches),
            "xent": float(xent), "htri": float(htri), "acc": float(acc),
        })
        continue
    m = MAP_RE.search(line)
    if m:
        final_map = float(m.group(1))
        continue
    m = RANK_RE.search(line)
    if m:
        cmc[int(m.group(1))] = float(m.group(2))

print(f"Parsed {len(batch_stats)} batch entries")
if final_map is not None:
    print(f"mAP       : {final_map:.1f}%")
for r, v in sorted(cmc.items()):
    print(f"Rank-{r:<3}  : {v:.1f}%")

# ── Per-epoch summary (take the running avg from the last batch of each epoch) ─
epoch_groups = defaultdict(list)
for s in batch_stats:
    epoch_groups[s["epoch"]].append(s)

epoch_summaries = []
for epoch, batches in sorted(epoch_groups.items()):
    last = batches[-1]
    epoch_summaries.append({
        "epoch":  epoch,
        "xent":   last["xent"],
        "htri":   last["htri"],
        "loss":   last["xent"] + last["htri"],
        "acc":    last["acc"],
    })

Path(save_dir).mkdir(parents=True, exist_ok=True)
plot_paths = []

# ── Plot 1: Training curves ───────────────────────────────────────────────────
if epoch_summaries:
    epochs = [s["epoch"] for s in epoch_summaries]

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    fig.suptitle(f"{arch}  lr={learning_rate}  epochs={max_epochs}", fontsize=12)

    axes[0].plot(epochs, [s["xent"] for s in epoch_summaries],
                 "o-", label="XEnt",    color="#2196F3")
    axes[0].plot(epochs, [s["htri"] for s in epoch_summaries],
                 "s-", label="Triplet", color="#FF5722")
    axes[0].plot(epochs, [s["loss"] for s in epoch_summaries],
                 "^--", label="Total", color="#4CAF50", linewidth=2)
    axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss (running avg)")
    axes[0].legend(); axes[0].grid(True, alpha=0.3)
    axes[0].set_title("Training Loss per Epoch")

    axes[1].plot(epochs, [s["acc"] for s in epoch_summaries],
                 "o-", color="#9C27B0", linewidth=2)
    axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Top-1 Acc (%)")
    axes[1].grid(True, alpha=0.3)
    axes[1].set_title("Training Accuracy per Epoch")

    plt.tight_layout()
    p = Path(save_dir) / "training_curves.png"
    plt.savefig(p, dpi=120, bbox_inches="tight")
    plt.show()
    plt.close()
    plot_paths.append(str(p))
    print(f"\nSaved: {p}")
elif not evaluate_only:
    print("No batch training stats found in output (check print_freq setting).")

# ── Plot 2: Evaluation metrics ────────────────────────────────────────────────
if cmc or final_map is not None:
    ranks  = sorted(cmc)
    values = [cmc[r] for r in ranks]

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    fig.suptitle(f"Evaluation — {arch}  lr={learning_rate}", fontsize=12)

    # CMC curve
    if ranks:
        axes[0].plot(ranks, values, "o-", color="#2196F3", linewidth=2, markersize=8)
        axes[0].set_xlabel("Rank"); axes[0].set_ylabel("Matching Rate (%)")
        axes[0].set_title("CMC Curve"); axes[0].grid(True, alpha=0.3)
        axes[0].set_xticks(ranks)
        for x, y in zip(ranks, values):
            axes[0].annotate(f"{y:.1f}%", (x, y), textcoords="offset points",
                             xytext=(0, 8), ha="center", fontsize=9)

    # Bar chart: Rank-1/5/10/20 + mAP
    bar_labels = [f"Rank-{r}" for r in ranks]
    bar_values = list(values)
    bar_colors = ["#2196F3"] * len(ranks)
    if final_map is not None:
        bar_labels.append("mAP")
        bar_values.append(final_map)
        bar_colors.append("#FF5722")

    bars = axes[1].bar(bar_labels, bar_values, color=bar_colors, edgecolor="white", linewidth=0.5)
    axes[1].set_ylabel("Score (%)"); axes[1].set_title("Evaluation Metrics")
    axes[1].set_ylim(0, max(bar_values) * 1.18 if bar_values else 100)
    for bar, v in zip(bars, bar_values):
        axes[1].text(bar.get_x() + bar.get_width() / 2,
                     bar.get_height() + 0.5,
                     f"{v:.1f}%", ha="center", fontsize=9)
    axes[1].grid(True, axis="y", alpha=0.3)

    plt.tight_layout()
    p = Path(save_dir) / "eval_metrics.png"
    plt.savefig(p, dpi=120, bbox_inches="tight")
    plt.show()
    plt.close()
    plot_paths.append(str(p))
    print(f"Saved: {p}")
else:
    print("No evaluation metrics found in output.")

# ── Save metrics JSON for the comparison notebook ─────────────────────────────
metrics = {
    "config": {
        "arch":             arch,
        "learning_rate":    learning_rate,
        "max_epochs":       max_epochs,
        "optimizer":        optimizer,
        "train_batch_size": train_batch_size,
        "image_height":     image_height,
        "image_width":      image_width,
        "label_smooth":     label_smooth,
        "random_erase":     random_erase,
        "color_jitter":     color_jitter,
        "margin":           margin,
        "save_dir":         save_dir,
    },
    "training": {
        "epochs":           epoch_summaries,
        "elapsed_seconds":  round(elapsed, 2),
    },
    "evaluation": {
        "map":  final_map,
        "cmc":  {str(k): v for k, v in cmc.items()},
    },
    "plots": plot_paths,
}

metrics_path = Path(save_dir) / "metrics.json"
metrics_path.write_text(json.dumps(metrics, indent=2))
print(f"\nMetrics JSON saved to {metrics_path}")

## 6. Output Files

Checkpoints, logs, plots and `metrics.json` are all written to `save_dir`.
The HTML export is generated by `scripts/run_lr_sweep.py` after Papermill finishes.

In [ ]:
log_dir = Path(save_dir)
if log_dir.exists():
    files = sorted(log_dir.rglob("*"))
    print(f"Output files in {log_dir}:")
    for f in files:
        if f.is_file():
            print(f"  {f.relative_to(log_dir):<40}  {f.stat().st_size:>10,} bytes")
else:
    print(f"Log directory not yet created: {log_dir}")

print(f"\nRun summary:")
print(f"  arch={arch}, lr={learning_rate}, epochs={max_epochs}, optim={optimizer}, seed={seed}")
print(f"  image={image_height}x{image_width}, batch={train_batch_size}")
print(f"  aug: random_erase={random_erase}, color_jitter={color_jitter}")
print(f"  loss: margin={margin}, label_smooth={label_smooth}")